# 2 — Clustering signals: which one recovers which axis?

The central question: can the server group clients by **domain** without seeing raw data?

Four candidate signals are compared, each scored against **both** ground truths — rotation group (feature axis) and label histogram (label axis). A good feature-axis signal scores high on the first and ~0 on the second.

**This notebook reproduces the main finding and needs no GPU and no training** — the winning signal is a forward pass through a *frozen, untrained* network.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # import hefl from the repo root

import numpy as np, torch, matplotlib.pyplot as plt
from hefl.utils import pick_device, set_seed
set_seed(42); DEVICE = pick_device('auto')
print('device:', DEVICE)


In [ ]:
from hefl.datasets import build_federated_data
from hefl.models import CifarBackbone
from hefl.clustering import activation_statistics, cluster_clients, label_axis_ground_truth
from sklearn.metrics import adjusted_rand_score
from torch.utils.data import DataLoader

data = build_federated_data(root='../data_cache', num_clients=24, alpha=0.5,
                            rotation_groups=[0, 1, 2, 3], seed=42)
backbone = CifarBackbone('gn').to(DEVICE).eval()   # RANDOM weights - never trained
label_truth = label_axis_ground_truth(data.client_label_hist, 4, 42)


## The spatial grid is the whole trick

`grid=1` averages each channel over the entire feature map. A rotated image has nearly the same global channel means, so that descriptor is blind to rotation and what survives is class content.

`grid=2` keeps a coarse spatial layout — *sky at the top* vs *sky on the left* — which is a direct signature of the domain.


In [ ]:
rows = []
for grid in (1, 2, 4):
    M = np.array([activation_statistics(backbone, DataLoader(ds, batch_size=64), DEVICE,
                                        max_batches=8, grid=grid)
                  for ds in data.client_sets])
    out = cluster_clients(M, 4, seed=42, pca_dim=16)
    rows.append(dict(grid=grid, dim=M.shape[1],
                     ari_rotation=adjusted_rand_score(data.client_rotation, out['labels']),
                     ari_label=adjusted_rand_score(label_truth, out['labels']),
                     silhouette=out['silhouette']))

print(f"{'grid':>5}{'dim':>7}{'ARI rotation':>15}{'ARI label':>12}{'silhouette':>13}")
for r in rows:
    print(f"{r['grid']:>5}{r['dim']:>7}{r['ari_rotation']:>15.3f}{r['ari_label']:>12.3f}{r['silhouette']:>13.3f}")


## Is it stable, and where does it break?

Sweep the label-skew strength. The signal should stay pinned to the feature axis — until label skew becomes extreme enough that a client's *class content* alone changes its activation statistics.


In [ ]:
res = []
for alpha in (100.0, 1.0, 0.5, 0.1):
    d = build_federated_data(root='../data_cache', num_clients=24, alpha=alpha,
                             rotation_groups=[0,1,2,3], seed=42, download=False)
    lt = label_axis_ground_truth(d.client_label_hist, 4, 42)
    for grid in (1, 2):
        M = np.array([activation_statistics(backbone, DataLoader(ds, batch_size=64), DEVICE,
                                            max_batches=8, grid=grid) for ds in d.client_sets])
        o = cluster_clients(M, 4, seed=42, pca_dim=16)
        res.append((alpha, grid, adjusted_rand_score(d.client_rotation, o['labels'])))

fig, ax = plt.subplots(figsize=(6.5, 4))
for grid, style in ((1, 'o--'), (2, 'o-')):
    xs = [r[0] for r in res if r[1] == grid]; ys = [r[2] for r in res if r[1] == grid]
    ax.semilogx(xs, ys, style, label=f'grid={grid}', linewidth=2, markersize=7)
ax.set_xlabel('Dirichlet α  (left = more label skew)'); ax.set_ylabel('ARI vs rotation ground truth')
ax.set_title('Global pooling collapses under label skew; a 2×2 grid does not')
ax.axhline(1.0, color='gray', ls=':', lw=1); ax.set_ylim(-0.05, 1.08)
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


## Compare against the alternatives (this part does train)

`delta_l4fc` is the classic choice — the update of `layer4 + fc` after local warmup. It is 8.4M-dimensional and, as the table shows, needs far more local data than a federated warmup provides.

Skip this cell if you only want the headline result.


In [ ]:
from hefl.clustering import run_warmup, clustering_report

warm = run_warmup(data.client_sets, DEVICE, fedavg_rounds=1, local_epochs=2,
                  signals=('delta_l4fc', 'delta_stem', 'act_stats'), act_batches=8)
rep = clustering_report(warm, data.client_rotation, data.client_label_hist,
                        num_clusters=4, seed=42, pca_dim=16)
print(f"{'signal':<14}{'dim':>10}{'ARI rotation':>15}{'ARI label':>12}")
for name, m in rep.items():
    print(f"{name:<14}{m['dim']:>10}{m['ari_rotation']:>15.3f}{m['ari_label']:>12.3f}")
